In [1]:
import pandas as pd  # Para manipulación de datos
import matplotlib.pyplot as plt  # Para visualización
import seaborn as sns  # Para gráficos más estilizados
import numpy as np
import os

In [2]:
# Cargar los datos
csv_file_path = r"../../data/limpieza/cleaned_data.csv"
df = pd.read_csv(csv_file_path)

df.head()

,timestamp,open_binance,high_binance,low_binance,close_binance,volume_binance,trades_binance,volumen_binance,SMA_50_binance,SMA_200_binance,...,volatilidad_200_dxy,momentum_50_dxy,momentum_200_dxy,var_diaria_dxy,rango_dia_dxy,RSI_dxy,pct_cambio_7d,tendencia_futura_7d,pct_cambio_30d,tendencia_futura_30d
0,2023-01-01,16541.77,16628.00,16499.01,16616.75,96925.41374,3218355.0,1.610585e+09,16782.6976,19703.90840,...,2.899194,-2.393339,-1.140007,0.240917,1.049999,39.121295,NaN,0,NaN,0
1,2023-01-02,16617.17,16799.23,16548.70,16672.87,121888.57191,4036118.0,2.032232e+09,16789.5580,19685.26620,...,2.894822,-2.266673,0.639999,0.240338,1.219999,43.440167,NaN,0,NaN,0
2,2023-01-03,16672.78,16778.40,16605.28,16675.18,159541.53733,5097596.0,2.660384e+09,16790.6724,19666.29805,...,2.895847,-2.140007,-0.180000,0.239762,1.389999,59.385586,NaN,0,NaN,0
3,2023-01-04,16675.65,16991.87,16652.66,16850.36,220362.18862,6310703.0,3.713182e+09,16789.6682,19655.69590,...,2.898148,-2.150002,-0.384998,-0.258321,0.769997,51.494959,NaN,0,NaN,0
4,2023-01-05,16850.36,16879.82,16753.00,16831.85,163473.56641,4842014.0,2.751563e+09,16793.0500,19636.98515,...,2.895634,-1.239998,0.470001,0.757795,1.279999,58.640238,NaN,0,NaN,0


# Eliminación de variables redundantes o con información duplicada


## 1. Binance – precios OHLC
**Eliminar:**
- open_binance
- high_binance
- low_binance
- volumen_binance: se calcula close_binance * volume_binance

**Dejar:**
- close_binance
- volume_binance
- trades_binance
- rango_dia_binance: diferencia de high y low
- var_diaria_binance: cambio porcentual entre un valor y el valor inmediatamente anterior (en porcentaje)

In [5]:
df.drop(columns=['open_binance', 'high_binance', 'low_binance', 'volumen_binance'], inplace=True)

## 2. Binance - Medias móviles
Mantener una combinación de corto y largo plazo, evitando duplicar SMA y EMA para ambos.

**Eliminar:**
- SMA_50_binance
- EMA_200_binance

**Dejar:**
- EMA_50_binance
- SMA_200_binance

In [7]:
df.drop(columns=['SMA_50_binance', 'EMA_200_binance'], inplace=True)

## 3. Binance - Otras varibles
Conservaremos las variables de volatilidad, momentum y RSI

## 4. Indicadores
Conservaremos los indicadores: `fear_greed_index_fg` y `bitcoin_gtrends`

## 5. Hashrate
Conservaremos la variable `hashrate_hashrate` y mantendremos una combinación de corto y largo plazo para las medias móviles como en Binance

**Eliminar:**
- SMA_50_hashrate
- EMA_200_hashrate

**Dejar:**
- EMA_50_hashrate
- SMA_200_hashrate

In [9]:
df.drop(columns=['SMA_50_hashrate', 'EMA_200_hashrate'], inplace=True)

## 6. Mercado Derivados Binance, S&P 500, Nasdaq, DXY
Al compartir las mismas variables que Binance, repetiremos el proceso:

**Eliminar:**
- open
- high
- low
- SMA_50
- EMA_200

**Dejar:**
- close
- volume
- trades
- var_diaria
- rango dia
- volatilidad (50/200)
- momentum (50/200)
- SMA_200
- EMA_50

In [11]:
# mDbinance
df.drop(columns=[
    'mDopen_mDbinance',
    'mDhigh_mDbinance',
    'mDlow_mDbinance',
    'mDSMA_50_mDbinance',
    'mDEMA_200_mDbinance'
        ],
        inplace=True)

# S&P 500, Nasdaq, DXY
sufijos = ['_sp500', '_nasdaq', '_dxy']
columnas = ['Open', 'High', 'Low', 'SMA_50', 'EMA_200']
columnas_a_eliminar = [col + suf for col in columnas for suf in sufijos]

df.drop(columns=columnas_a_eliminar, inplace=True)

## 7. Varibles respuesta categóricas
Para mostrar la creación de la variable respuesta gráficamente se creó una variable categórica auxiliar que indicaba la tendencia alcista o bajista del bitcoin. Como no se va a realizar problemas de clasificación se eliminarán estas variables

In [13]:
df.drop(columns=['tendencia_futura_7d', 'tendencia_futura_30d'], inplace=True)

## 8. Conjunto de datos para 7 días
No se eliminará ninguna otra variable salvo la variable respuesta para 30 días

In [29]:
# Eliminamos la columna asociada al otro horizonte (30 días)
df_7d = df.drop(columns=['pct_cambio_30d'])

# Guardamos las últimas 7 filas (sin variable de respuesta) para hacer predicciones futuras
pred_7d = df_7d.tail(7).copy()

# Eliminamos las filas con valores nulos (generados por rolling y shift)
df_7d = df_7d.dropna()

#Guardamos los csv
output_path = r"../../data/limpieza/dataset_7d.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df_7d.to_csv(output_path, index=False)

output_path = r"../../data/limpieza/pred_7d.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
pred_7d.to_csv(output_path, index=False)

In [31]:
df_7d.head()

,timestamp,close_binance,volume_binance,trades_binance,SMA_200_binance,EMA_50_binance,var_diaria_binance,volatilidad_50_binance,volatilidad_200_binance,rango_dia_binance,...,SMA_200_dxy,EMA_50_dxy,volatilidad_50_dxy,volatilidad_200_dxy,momentum_50_dxy,momentum_200_dxy,var_diaria_dxy,rango_dia_dxy,RSI_dxy,pct_cambio_7d
6,2023-01-07,16943.57,104526.56880,3227161.0,19599.96920,17107.863305,-0.041768,361.625444,2272.933240,73.91,...,107.882558,105.269143,1.086780,2.905359,-3.343335,-0.853338,-0.282376,1.506663,42.849748,10.107373
7,2023-01-08,17127.83,135155.89695,4036092.0,19585.66840,17108.646312,1.087492,364.193483,2279.467987,265.99,...,107.878025,105.191660,1.063861,2.911834,-3.940000,-0.906665,-0.283176,1.253332,40.654665,12.801884
8,2023-01-09,17178.26,266211.52723,6327667.0,19566.00905,17111.376261,0.294433,359.567007,2283.206355,294.14,...,107.870875,105.105713,1.028797,2.922081,-4.536664,-1.430000,-0.283980,1.000000,38.673519,15.701148
9,2023-01-10,17440.66,221382.42581,5448751.0,19547.02390,17124.289349,1.527512,336.687863,2285.021937,352.66,...,107.866125,105.032547,0.957006,2.928859,-4.599998,-0.950005,0.233008,0.459999,41.546737,18.054824
10,2023-01-11,17943.26,262221.60653,5627464.0,19529.28425,17156.405845,2.881772,357.395836,2283.624315,684.40,...,107.861542,104.960291,0.912070,2.935481,-4.029999,-0.916667,-0.048426,0.370003,38.086322,19.253990


## 9. Conjunto de datos para 30 días
No se eliminará ninguna otra variable salvo la variable respuesta para 7 días

In [39]:
# Eliminamos la columna asociada al otro horizonte (7 días)
df_30d = df.drop(columns=['pct_cambio_7d'])

# Guardamos las últimas 30 filas (sin variable de respuesta) para hacer predicciones futuras
pred_30d = df_30d.tail(30).copy()

# Eliminamos las filas con valores nulos (generados por rolling y shift)
df_30d = df_30d.dropna()

#Guardamos los csv
output_path = r"../../data/limpieza/dataset_30d.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df_30d.to_csv(output_path, index=False)

output_path = r"../../data/limpieza/pred_30d.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
pred_30d.to_csv(output_path, index=False)

In [10]:
df.head()

,timestamp,close_binance,volume_binance,trades_binance,SMA_200_binance,EMA_50_binance,var_diaria_binance,volatilidad_50_binance,volatilidad_200_binance,rango_dia_binance,...,EMA_50_dxy,volatilidad_50_dxy,volatilidad_200_dxy,momentum_50_dxy,momentum_200_dxy,var_diaria_dxy,rango_dia_dxy,RSI_dxy,pct_cambio_7d,pct_cambio_30d
0,2023-03-15,24285.66,581450.72984,12613034.0,19760.26415,22364.086769,-1.559561,1105.739547,2503.541768,1300.02,...,104.123541,1.095841,3.647272,2.730003,-4.163333,1.013516,1.659996,51.504406,21.990165,21.648551
1,2023-03-16,24998.78,439421.32998,10418081.0,19787.48000,22467.408072,2.936383,1140.879297,2530.744205,1044.40,...,104.135166,1.056470,3.648377,2.779999,-4.406667,-0.219783,0.540001,44.277686,20.521633,21.919984
2,2023-03-17,27395.13,624460.68091,13113359.0,19823.02700,22660.652069,9.585868,1295.696250,2587.080336,2866.84,...,104.118493,1.017897,3.651955,1.870003,-5.129997,-0.679946,0.759995,42.676352,15.357288,21.800719
3,2023-03-18,26907.49,371238.97174,9948603.0,19858.50615,22827.194733,-1.780024,1400.927082,2635.133401,1146.85,...,104.096853,0.980797,3.656159,1.636665,-5.203331,-0.138206,0.733330,42.017841,11.407531,21.572368
4,2023-03-19,27972.87,372066.99054,9578297.0,19898.12040,23028.985920,3.959418,1553.615915,2696.856579,1562.88,...,104.070440,0.947178,3.661000,1.376666,-5.276665,-0.138397,0.706665,41.379343,7.713328,21.327590
